# Runnable Message History

The `history.py` module defines a Runnable wrapper that reads conversation messages from a `BaseChatMessageHistory`, passes them to another Runnable, and writes the new input and output messages back to that history.

The history instance is selected through configurable values supplied at runtime. By default, the wrapper expects a shared string configuration field named `session_id`, but custom history-factory fields may also be defined.

## Type Aliases

1. `MessagesOrDictWithMessages`: Represents either a sequence of chat messages or a dictionary containing message-related values.
   * **Definition:**
     ```python
     MessagesOrDictWithMessages = Sequence[
         BaseMessage
     ] | dict[
         str,
         Any
     ]
     ```

2. `GetSessionHistoryCallable`: Represents a callable that returns a `BaseChatMessageHistory`.

   The callable may accept one session identifier or multiple keyword arguments corresponding to `history_factory_config`.

   * **Definition:**
     ```python
     GetSessionHistoryCallable = Callable[
         ...,
         BaseChatMessageHistory
     ]
     ```

# RunnableWithMessageHistory

`RunnableWithMessageHistory` wraps another Runnable and manages its chat-message history.

Before execution, it obtains the appropriate history instance and supplies existing messages to the wrapped Runnable. After successful execution, it adds the current input messages and generated output messages to the same history.

The wrapped Runnable may accept a message sequence directly, a dictionary containing all messages, or a dictionary containing separate current-input and history fields. Its output may be a string, one message, a sequence of messages, or a dictionary containing messages.

This class is deprecated since version `1.3.3`, is scheduled for removal in `2.0.0`, and recommends using LangGraph persistence instead.

## Bases

- `RunnableBindingBase[Any, Any]`

## Attributes

1. `get_session_history`: Stores the factory used to obtain a chat-message history.

   With the default configuration, the callable receives one positional `session_id`. With custom `history_factory_config`, its parameter names must match the configurable-field identifiers.

   * **Type:**
     ```python
     get_session_history: GetSessionHistoryCallable
     ```

2. `input_messages_key`: Stores the input-dictionary key containing the current message or messages.

   It must be supplied when the wrapped Runnable accepts dictionary input and the messages are stored under a particular key.

   * **Type:**
     ```python
     input_messages_key: str | None = None
     ```

3. `output_messages_key`: Stores the output-dictionary key containing the generated message or messages.

   It must be supplied when the wrapped Runnable returns a dictionary with messages under a particular key.

   * **Type:**
     ```python
     output_messages_key: str | None = None
     ```

4. `history_messages_key`: Stores the input-dictionary key used specifically for historical messages.

   When this field is provided, historical messages and the current input are supplied to the wrapped Runnable through separate dictionary fields.

   * **Type:**
     ```python
     history_messages_key: str | None = None
     ```

5. `history_factory_config`: Stores the configurable-field specifications passed to the history factory.

   When no custom specifications are provided, one shared string field named `session_id` is created automatically.

   * **Type:**
     ```python
     history_factory_config: Sequence[
         ConfigurableFieldSpec
     ]
     ```

### Properties

1. `config_specs`: Returns the unique configuration specifications exposed by the wrapped Runnable and the history factory.
   * **Type:**
     ```python
     config_specs: list[
         ConfigurableFieldSpec
     ]
     ```

2. `OutputType`: Returns the output type exposed by the internally constructed history-loading chain.
   * **Type:**
     ```python
     OutputType: type[Output]
     ```

### Methods

1. `__init__`: Creates a message-history wrapper around a Runnable or language model.

   The constructor builds a history-loading Runnable, optionally inserts the loaded history under a dictionary key, attaches synchronous and asynchronous completion listeners, and binds the resulting chain.

   The wrapped Runnable must accept either a sequence of messages or a compatible message dictionary. Its output must be convertible into AI messages or message sequences.

   * **Syntax:**
     ```python
     __init__(
         self,
         runnable: Runnable[
             list[BaseMessage],
             str
             | BaseMessage
             | MessagesOrDictWithMessages
         ]
         | Runnable[
             dict[str, Any],
             str
             | BaseMessage
             | MessagesOrDictWithMessages
         ]
         | LanguageModelLike, # Runnable or language model to wrap
         get_session_history: GetSessionHistoryCallable, # Factory that returns the relevant message history
         *,
         input_messages_key: str | None = None, # Input key containing current messages
         output_messages_key: str | None = None, # Output key containing generated messages
         history_messages_key: str | None = None, # Separate input key for historical messages
         history_factory_config: Sequence[
             ConfigurableFieldSpec
         ] | None = None, # Configuration fields passed to the history factory
         **kwargs: Any # Additional RunnableBindingBase arguments
     ) -> None
     ```

2. `get_input_schema`: Returns a Pydantic model describing the accepted input format.

   When both `input_messages_key` and `history_messages_key` are configured, the current-input field accepts a string, one message, or a sequence of messages. When only `input_messages_key` is configured, that field accepts a message sequence. Without an input key, the schema uses a root message sequence.

   * **Syntax:**
     ```python
     get_input_schema(
         self,
         config: RunnableConfig | None = None # Runtime configuration used for schema generation
     ) -> type[BaseModel]
     ```

3. `get_output_schema`: Returns a Pydantic model describing the wrapper's output.

   When `OutputType` is already a Pydantic model class, that class is returned directly. Otherwise, a root model named `RunnableWithChatHistoryOutput` is created for the output type.

   * **Syntax:**
     ```python
     get_output_schema(
         self,
         config: RunnableConfig | None = None # Runtime configuration used for schema generation
     ) -> type[BaseModel]
     ```

## History Processing

Before synchronous or asynchronous execution, the wrapper obtains the `BaseChatMessageHistory` stored in the merged configuration and loads its existing messages.

When `history_messages_key` is not configured, the current input messages are appended to the loaded history before the wrapped Runnable is called. When it is configured, the history is supplied separately and the current input remains under `input_messages_key`.

After execution, the wrapper converts string outputs into `AIMessage` objects, normalizes individual messages and message sequences, removes already-prepended historical messages when necessary, and appends only the new input and output messages to the history.

## Configuration Validation

The wrapper verifies that all fields defined by `history_factory_config` are present in `config["configurable"]`.

When exactly one field is expected, its value is passed positionally to a history factory that declares parameters. A zero-parameter factory is called without arguments.

When multiple fields are expected, their identifiers must exactly match the history factory's parameter names, and the values are passed as keyword arguments.

A `ValueError` is raised when required configuration values are missing or when custom field identifiers do not match the history factory's parameter names.